# 对比与判读

训练在 [`MSN_train_skullfix.ipynb`](MSN_train_skullfix.ipynb)，这里只判读。

1 选 run · **2 指标词典** · **3 判决标准** · 4 曲线 · **5 同轮次表** · 6 主表 · **7 配对检验** · 8 分布图 · 9 可视化 · 10 对照组

> ⚠️ 第 6 节起占显存。**回去训练前必须 Restart Kernel。**

## 1. 选 run

`RUNS` 加一行，下面所有表和图就多一列；缺权重的自动跳过。

⛔ 不要加回 `baseline_es20` / `dcd_w3` / `dcd_l2` / `rep05_void` —— 学习率衰减从未触发，数字不可读，权重也已删。

In [1]:
import os, sys, json
import numpy as np
import pandas as pd

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
sys.path.insert(0, os.path.join(REPO, "src", "eval"))
sys.path.insert(0, os.path.join(REPO, "src", "models"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("HF_HOME", "/root/.cache/huggingface")

import importlib, report as rp
importlib.reload(rp)      # 改过 report.py 之后必须 reload，否则拿到的是缓存的旧模块

# ============================ 要比哪几个 ============================
RUNS = [
    ("cd_only",       "msn_skullfix/cd_only"),        # 2x2: CD 单独
    ("lr_fix",        "msn_skullfix/lr_fix_only"),    # 2x2: CD+DCD
    ("rep_w05",       "msn_skullfix/rep_w05"),        # 2x2: CD+DCD+rep
    ("cd_rep05_full", "msn_skullfix/cd_rep05_full"),  # 2x2: CD+rep  ← 目前的最优配置
    ("cd_rep05_r2",   "msn_skullfix/cd_rep05_r2"),    # 同配置重复 → 这是噪声底线的来源
    ("tie_qk",        "msn_skullfix/tie_qk"),         # Q/K 初始绑定
    ("tie_qk_r2",     "msn_skullfix/tie_qk_r2"),      # 它的重复实验（跑完自动出现）
    ("notext",        "msn_skullfix/notext"),         # 去掉文本分支
    ("pp_attn",       "msn_skullfix/pp_attn"),        # 逐点交叉注意力 ❌ 已否决
]
BASE = "cd_rep05_full"     # 第 7 节配对检验的比较基准
# ===================================================================

_missing = [(n, p) for n, p in RUNS
            if not os.path.exists(os.path.join(REPO, "experiments", p, "best.h5"))]
if _missing:
    print("无权重，已跳过（未跑，或已按有效性分界裁剪）:",
          ", ".join(n for n, _ in _missing), "\n")
runs = rp.load_runs(REPO, [r for r in RUNS if r not in _missing])

print(f"{'run':16}{'配置':34}{'轮数':>6}{'停止':>16}{'LR降':>6}{'CD_t(run.json)':>16}")
for r in runs:
    n_ep, best_ep = r.meta["epochs_run"], int(r.hist["val_loss"].idxmin()) + 1
    # 先判早停：它是决定性的（最后 patience 轮没有改善）。反过来先判上限会误伤 ——
    # 早期的 run.json 没记 --epochs，回退值可能比它实际用的上限小。
    if n_ep - best_ep == r.meta["early_stop_patience"]:
        stop = "EarlyStopping"
    elif "epochs" not in r.meta:
        stop = "⚠️ 上限未记录"
    elif n_ep >= r.meta["epochs"]:
        stop = "❌ 被上限截断"
    else:
        stop = "⚠️ 墙钟掐停"
    print(f"{r.label:16}{r.config_str():34}{n_ep:>6}{stop:>16}"
          f"{len(r.lr_drops):>6}{r.meta['best_val_cd_t_mm']:>16.3f}")

assert len({tuple(r.meta["val_ids"]) for r in runs}) == 1, "各 run 的验证集划分不同，不可比"
print(f"\n所有 run 共用同一批 {len(runs[0].meta['val_ids'])} 颗验证颅骨 ✅")
print("架构:", ", ".join(sorted({r.arch_label for r in runs})))

run             配置                                    轮数              停止   LR降  CD_t(run.json)
cd_only         loss=cd                              305   EarlyStopping     8           6.353
lr_fix          loss=cd_dcd  λ=2                     279   EarlyStopping     7           6.317
rep_w05         loss=cd_dcd  λ=2  rep=0.5@2mm        222   EarlyStopping     6           6.326
cd_rep05_full   loss=cd  rep=0.5@2mm                 256   EarlyStopping     9           6.267
cd_rep05_r2     loss=cd  rep=0.5@2mm                 249   EarlyStopping     7           6.274
tie_qk          loss=cd  rep=0.5@2mm                 411   EarlyStopping     9           6.198
tie_qk_r2       loss=cd  rep=0.5@2mm                 246   EarlyStopping     6           6.319
notext          loss=cd  rep=0.5@2mm                 380   EarlyStopping     9           6.225
pp_attn         loss=cd  rep=0.5@2mm                 355   EarlyStopping     9           6.581

所有 run 共用同一批 20 颗验证颅骨 ✅
架构: paper, paper+no_text,

## 2. 指标词典

**主指标是 `defect_cov_mm`（缺损区覆盖）。** 只有 6.7% 的 GT 点落在缺损区，其余 93.3% 是输入里已给、模型只需复现的表面 —— 全点云指标主要在量"抄得像不像"。实测佐证：2×2 消融在缺损覆盖上**四格全部显著**，在全点云 CD_t 上**全部不显著**。

### 缺损区（主表）

| 列 | 是什么 | 能被糊弄吗 |
|---|---|---|
| **`defect_cov_mm`** ⭐ | 洞里每个 GT 点到最近预测点的距离 = **填得全不全** | ✅ 不能（不填洞直接爆到 12.94） |
| `defect_HD95_mm` | 同上的 95 分位 | ✅ 不能 |
| `defect_prec_mm` | 放进洞里的点离真实表面多远 = 填得准不准 | ⚠️ **能**（一个点不放就是 nan），要连 `defect_n_pred` 一起看。且**在损失函数这一维上不区分模型**（2.89~3.01） |
| `defect_n_pred` | 放进洞里的点数 | GT 约 395，各配置 387~492 |
| `defect_gt_%` | 缺损区占 GT 的比例 | 数据属性，各 run 相同，不用比 |

判定规则两侧**不同**（刻意的）：GT 侧「到最近**输入点** > 5mm」，预测侧「到最近**缺损 GT 点** < 5mm」。5mm 取自 GT→输入距离双峰分布的谷底。

### 全点云（辅表）

| 列 | 什么时候用 |
|---|---|
| `CD_t_mm` | 通用精度。**别单独用它下结论**（93.3% 权重在复现输入上） |
| `HD95_mm` | 最坏情况，临床角度（CD 会把"某处差 15mm"平均掉） |
| `F1@0.05` / `@0.03` | 和原论文 Table 1/2 并排的两个之一（≈5.19 / 3.11mm） |
| `DCD` | 另一个可跨项目比的；也是复现正确性的证据（论文 1.41269 vs 实测 1.42772） |
| `clump_%` | 密度直接读数（GT = 0.0%），repulsion 就冲它去的 |
| `spacing_CV` | 同上但更平滑（GT = 0.145） |

不报 `CD_p`：定义是 `sqrt(平均距离)`，对长度开根号，量纲不成立。

### 什么时候盯哪个 ⭐

| 你改的是 | 盯 |
|---|---|
| 损失函数 / 密度项 | `clump_%` + `spacing_CV`（效应 20/20 一致，最灵敏） |
| 架构 / 初始化 | `defect_cov_mm`（这类效应只有 0.02~0.09mm，全点云分辨不出） |
| 想说"**补全能力**变强了" | 只有 `defect_cov_mm` 算数 |
| 和**原论文**并排 | `DCD`、`F1`；⚠️ `CD` 不可跨项目比 |
| 离**临床**多远 | `HD95` + Poisson 重建（未做） |

**采样地板**：CD_t 4.43mm、HD95 3.79mm，单向约 2.2mm。缺损覆盖 3.24mm 里约 2.2mm 是采样分辨率，**模型自身只贡献约 1mm**。⚠️ 本文数值**不可与体素域方法并排**。

## 3. 判决标准

### 三把尺子，回答三个不同的问题

| 尺子 | 回答什么 | 当前值 |
|---|---|---|
| ① **同配置重跑** | 重训一次数字会不会变 | **看下面那张表** —— 三对重复实验，两对 ≲0.005mm，一对 0.15mm |
| ② **逐颅骨配对**（第 7 节） | 换一批颅骨还成不成立 | 颅骨间 CD_t std ≈ 1.0mm，配对后 SE ≈ 0.06mm |
| ③ **末段抖动** | 报告的最优值本身抖多少 | 退火后 0.003~0.010mm；没退火 ~0.25mm |

### ① 的三对实测 —— 大小取决于**曲线走没走平**

| 重复对 | CD_t | 缺损覆盖 | 轮数 | 末段还在降吗 |
|---|---:|---:|---|---|
| `cd_rep05` 那对 | 0.0040 | 0.0065 | 256 / 249 | 已走平 |
| **`tie_qk` 那对** | **0.1506** | **0.2328** | 411 / 246 | **还在降** |
| `notext` 那对 | 0.0013 | 0.0037 | 380 / 227 | 已走平 |

`notext` 那对**轮数差了 153 轮**却复现到 0.0013mm —— 所以"轮数差得多"本身**不**导致大方差。
**判据：读小差异之前，先看第 5 节同轮次表的末几列有没有走平。** 还在明显下行的 run，
它的"全程最优"不能直接拿去比。（`clump_%` 另算：0.2pp 以下不可信。）

⚠️ **①② 互不替代**：`tie_qk` 两次训练之间的差异是**跨颅骨一致**的（2/20，p=0.0004）——
所以**配对检验能证明"两个模型不同"，不能证明"这个配置更好"**。

⚠️ 另一个坑：**early stopping 盯 `val_loss`，而结论读 `val_cd_t` / 缺损覆盖。**
"按停止准则收敛了"≠"按报告指标收敛了"（`lr_fix_only`、`cd_only`、`rep_w05` 都属这档）。

### 一个改动算"成立"，要同时满足

- [ ] **同轮次表上仍领先**（不是靠多跑几十轮）
- [ ] **配对检验主指标** `p_wilcoxon < 0.002`，或改善 ≥ 17/20
- [ ] **有同配置重复**，且两次差异 < 声称的效应
- [ ] 方向与机制解释一致

不满足 → devlog 标 ⚠️ 待确认，**不进论文**。两个实例：`tie_qk` 四条全不过 → 否决；
`notext` 四条全过（重复差 0.0037 vs 效应 0.174，47 倍）→ 成立。

## 4. 训练曲线

虚线 = 学习率下降。**没有虚线的 run 直接作废** —— 那说明衰减从未触发。

第四格「val defect coverage」只有 2026-08-25 之后训练的 run 才有（`--defect-every`，
每 10 轮采一个点）。它是**诊断**：看主指标停下来时走平没有。**它不参与任何选择** ——
用它挑 checkpoint 会让报告值乐观偏置。

In [2]:
rp.fig_curves(runs).show()

## 5. 同轮次表 ⭐

各 run 由早停自己停，停在 222~411 轮不等，而报告的是**全程最优** —— 跑得久的天然占便宜。

- 在最短的共同轮次上**仍领先** → 是配置的功劳
- **只在自己的停止点领先** → 量到的是"这次跑得久"。这也可能是配置的真实性质，但那是**另一句话**，要靠重复实验分开

实例：`tie_qk` 报告领先 0.095mm，两边都截到 249 轮只剩 0.02~0.03mm；重复一次后整体反向。

In [3]:
at = sorted({222, 249, 300, min(len(r.hist) for r in runs)})
print(rp.epoch_matched(runs, at=at).to_string())
print("\n列的含义: reported = 全程最优（各 run 自己的停止点）；@N = 都截到第 N 轮时的最优；"
      "\nlate_std = 末 30 轮抖动（尺子③）。单位 mm，history 口径。")

               epochs  best_epoch  reported    @222    @249    @300  late_std
run                                                                          
cd_only           305         285    6.3531  6.4226  6.3733  6.3531    0.0050
lr_fix            279         277    6.3173  6.3408  6.3272     NaN    0.0071
rep_w05           222         220    6.3256  6.3256     NaN     NaN    0.0095
cd_rep05_full     256         236    6.2665  6.2797  6.2665     NaN    0.0026
cd_rep05_r2       249         229    6.2739  6.2856  6.2739     NaN    0.0095
tie_qk            411         391    6.1978  6.2732  6.2469  6.2140    0.0032
tie_qk_r2         246         226    6.3193  6.3289     NaN     NaN    0.0092
notext            380         360    6.2245  6.2427  6.2396  6.2329    0.0025
pp_attn           355         335    6.5806  6.6230  6.6024  6.5913    0.0049

列的含义: reported = 全程最优（各 run 自己的停止点）；@N = 都截到第 N 轮时的最优；
late_std = 末 30 轮抖动（尺子③）。单位 mm，history 口径。


## 6. 逐颅骨评估 —— 主表

**这一节起占显存**（每套架构建一个 187M 模型）。

推理是**完全确定性的**（验证走固定种子的 stateless 采样），所以只要权重还在，这张表随时逐位重算 —— 训练则不可复现。

In [4]:
eval_df = rp.eval_runs(REPO, runs)     # 每颗验证颅骨一行
print()
print("=== 缺损区（主表）===")
print(rp.format_defect_summary(eval_df))
print()
print("=== 全点云（辅表）===")
print(rp.format_summary(eval_df))

2026-08-24 02:19:02.305908: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-24 02:19:02.305928: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-24 02:19:02.306561: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[eval_runs] 4 architectures present, one model each: paper x5, paper x2, paper x1, paper+pp_attn x1
  cd_only: 20 skulls
  lr_fix: 20 skulls
  rep_w05: 20 skulls
  cd_rep05_full: 20 skulls
  cd_rep05_r2: 20 skulls
  tie_qk: 20 skulls
  tie_qk_r2: 20 skulls
  notext: 20 skulls
  pp_attn: 20 skulls

=== 缺损区（主表）===
run                    defect_cov_mm  defect_HD95_mm  defect_prec_mm  defect_F1@0.05     defect_gt_%   defect_n_pred
--------------------------------------------------------------------------------------------------------------------
cd_only                        3.439           5.500           2.928           0.962           6.436         424.700
lr_fix                         3.248           5.227           2.890           0.966           6.436         461.750
rep_w05                        3.414           5.539           2.958           0.957           6.436         451.150
cd_rep05_full                  3.236           5.312           2.913           0.965           6.436 

### 6.1 存档（可选）

`eval_all_runs.csv` 是跟踪进 git 的冻结记录，里面还有 `baseline` / `dcd_l2` 两行 —— 它们的权重已删、**再也算不出来**。所以这里按 run 名**合并**，不是覆盖。

In [ ]:
SAVE = False        # 确认表没问题之后改成 True

if SAVE:
    path = os.path.join(REPO, "experiments_log", "eval_all_runs.csv")
    merged = eval_df
    if os.path.exists(path):
        prev = pd.read_csv(path)
        merged = pd.concat([prev[~prev["run"].isin(eval_df["run"])], eval_df], ignore_index=True)
    merged.to_csv(path, index=False)
    print(f"-> {path}  ({merged['run'].nunique()} 个 run / {len(merged)} 行)")
else:
    print("SAVE=False，没有写盘。确认上面的表没问题后改成 True 再跑一次。")

## 7. 配对检验 ⭐

同一批 20 颗颅骨逐颗配对，消掉"这颗颅骨本身就难"（颅骨间 std ≈ 1.0mm，比要找的效应大一个数量级，不配对全会被淹没）。

- `delta` = 改动后 − 基准；`↓好` 的指标里负数是改善
- **`改善 18/20` 那一列常比 `delta` 更有说服力** —— 均值的置信区间往往很宽
- ⚠️ 检验做得多，用 **`p < 0.002`** 这条线，不是 0.05
- ⚠️ 它**看不到训练随机性**，必须配尺子①（重复实验）一起看

In [ ]:
for r in runs:
    if r.label == BASE:
        continue
    print(rp.format_paired(eval_df, BASE, r.label,
                           cols=["defect_cov_mm", "defect_HD95_mm", "defect_prec_mm",
                                 "CD_t_mm", "HD95_mm", "F1@0.05", "clump_%", "spacing_CV"]))
    print()

## 8. 分布图

均值会骗人。箱线图的**重叠程度**才是"这个差异是否可信"的直接证据。

In [ ]:
rp.fig_per_skull(eval_df, "defect_cov_mm").show()    # 主指标
rp.fig_per_skull(eval_df, "CD_t_mm").show()
rp.fig_progress(eval_df, baseline=BASE).show()       # 所有指标翻转成「向下 = 变好」

## 9. 可视化（点云）

蓝色 = 预测的完整颅骨，浅红 = 缺损输入，**缺损区应该只有蓝色**。

> 散点图只能看形状。看表面质量和疏密用 [`MSN_surface_quality.ipynb`](MSN_surface_quality.ipynb)。

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import tensorflow as tf
import msn_skullfix as msn

SHOW_RUN = BASE          # 改这里换 run

_run = next(r for r in runs if r.label == SHOW_RUN)
for _g in tf.config.experimental.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

# 架构由这个 run 自己的 run.json 决定，不要写死 paper()。改解码器 key 的来源或关掉
# 文本分支都会换一套拓扑，而前者不改变任何权重形状 —— 旧 checkpoint 会被静默读进
# 新拓扑、不报错，然后给出一份属于「从未训练过的网络」的图。
_cfg = rp.arch_config(msn, _run.arch_key)
model = msn.build_model(_cfg)
model.load_weights(_run.weights)

_data = np.load(os.path.join(REPO, "data", "cache", "skullfix_pairs_4096_6144.npz"))
ids, inputs, gt = _data["ids"], _data["inputs"], _data["gt"]
_val = _run.meta["val_ids"]
val_pos = [int(np.where(ids == v)[0][0]) for v in _val]
_x = [inputs[val_pos]]
if _cfg.use_text:
    _x.append(np.tile(np.load(os.path.join(REPO, "data", "cache", "bert_skull.npy"))[None],
                      (len(val_pos), 1)))
# predict 而不是 model(x) 逐个调用：后者实测每次泄漏 0.29 GiB 且不释放
preds = model.predict(_x, batch_size=1, verbose=0)
print(f"{SHOW_RUN} ({_run.arch_label}): {model.count_params()/1e6:.1f}M 参数, {len(preds)} 颗")

PRED, INP, GTC = rp.C_TRAIN, "#EF9A9A", rp.C_GT


def show_completion(k, camera=(1.6, 1.6, 1.2)):
    pred, pos, sid = preds[k], val_pos[k], _val[k]
    fig = go.Figure([
        go.Scatter3d(x=pred[:, 0], y=pred[:, 1], z=pred[:, 2], mode="markers",
                     name="Predicted complete skull",
                     marker=dict(size=1.6, color=PRED, opacity=0.85)),
        go.Scatter3d(x=inputs[pos][:, 0], y=inputs[pos][:, 1], z=inputs[pos][:, 2],
                     mode="markers", name="Defective input",
                     marker=dict(size=1.5, color=INP, opacity=0.45))])
    fig.update_layout(title=f"skull_{sid} ({SHOW_RUN}) — 缺损区应只有蓝色", height=650,
                      legend=dict(itemsizing="constant", x=0.02, y=0.98,
                                  bgcolor="rgba(255,255,255,0.6)"),
                      scene=dict(aspectmode="data", camera=dict(eye=dict(zip("xyz", camera)))),
                      margin=dict(l=0, r=0, b=0, t=40))
    return fig


def show_pred_vs_gt(k):
    pred, pos, sid = preds[k], val_pos[k], _val[k]
    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}] * 2],
                        subplot_titles=(f"Prediction ({SHOW_RUN})", "Ground truth"))
    fig.add_trace(go.Scatter3d(x=pred[:, 0], y=pred[:, 1], z=pred[:, 2], mode="markers",
                               name="Predicted", marker=dict(size=1.4, color=PRED)), 1, 1)
    g = gt[pos]
    fig.add_trace(go.Scatter3d(x=g[:, 0], y=g[:, 1], z=g[:, 2], mode="markers",
                               name="Ground truth", marker=dict(size=1.4, color=GTC)), 1, 2)
    fig.update_layout(height=520, title=f"skull_{sid}",
                      legend=dict(itemsizing="constant", orientation="h",
                                  x=0.5, xanchor="center", y=-0.02),
                      scene=dict(aspectmode="data"), scene2=dict(aspectmode="data"))
    return fig


# 缺损覆盖最好和最差的两颗 —— 看模型在哪种洞上撑不住
_d = eval_df[eval_df["run"] == SHOW_RUN]["defect_cov_mm"].reset_index(drop=True)
show_completion(int(_d.idxmin())).show()
show_pred_vs_gt(int(_d.idxmax())).show()

## 10. 对照组

### 作者发布的预训练权重

`experiments_log/pretrained_baseline/eval_val20.csv` —— 同一批 20 颗、同一套指标定义、同一份 `.npz`，对照干净。

- ⚠️ 措辞：这是**"通用基础模型直接应用于颅骨" vs "颅骨专精训练"**，不是同任务下两个方法的较量。论文报 DCD 1.41269、实测 1.42772（差 1%）→ 它在颅骨上**并未失效**，本工作的增益来自**专精化**
- ⚠️ 不要写 "zero-shot"（MedShapeNet 含 bones 类且部分源自 AutoImplant，很可能见过颅骨）
- ⚠️ 这个 CSV **没有缺损区列**，主表要并排得补跑一次推理（TODO 第 17 项）

### 原论文报告的数字

| | 训练数据 | CD | DCD | F1@0.05 | F1@0.03 |
|---|---|---:|---:|---:|---:|
| Table 1 | 200,000 点云 / 240 类 | 0.00170 | 1.41269 | 0.9370 | 0.7408 |
| Table 2 | 4,800 形状 | 0.002327 | 1.60467 | 0.89202 | 0.6234 |

Table 2 最接近本项目处境。他们 6×A6000 训六周，本项目单卡 40~60 分钟 —— **差三个数量级**，引用绝对数字必须交代。

⚠️ **`CD` 那列不可跨项目比**（0.00170 比点间距还小 35 倍，量纲对不上）；DCD 和 F1 没这问题。

In [ ]:
pre = pd.read_csv(os.path.join(REPO, "experiments_log", "pretrained_baseline", "eval_val20.csv"))
best_run = eval_df.groupby("run")["defect_cov_mm"].mean().idxmin()     # 按主指标挑
mine = eval_df[eval_df["run"] == best_run]

print(f"同样 {len(pre)} 颗验证颅骨，同样的指标定义\n")
print(f"{'':<46}{'CD_t (mm)':>12}{'DCD':>10}")
print("-" * 68)
print(f"{'作者发布的预训练权重（未在颅骨上训练）':<46}"
      f"{pre['CD_t_mm'].mean():>12.3f}{pre['DCD'].mean():>10.4f}")
print(f"{'本工作（' + best_run + '，从零训练）':<46}"
      f"{mine['CD_t_mm'].mean():>12.3f}{mine['DCD'].mean():>10.4f}")
print("-" * 68)
print(f"{'改善':<46}{(1 - mine['CD_t_mm'].mean() / pre['CD_t_mm'].mean()) * 100:>11.1f}%"
      f"{(1 - mine['DCD'].mean() / pre['DCD'].mean()) * 100:>9.1f}%")
print(f"\n复现性核对：论文 Table 1 报 DCD = 1.41269，这里实测 {pre['DCD'].mean():.5f}"
      f" → 差 {abs(pre['DCD'].mean() - 1.41269) / 1.41269 * 100:.1f}%。"
      f"\n指标实现与权重加载两件事同时被验证。")